In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import morethemes as mt
mt.set_theme('economist')
import warnings
warnings.filterwarnings('ignore')
import yfinance as yf
import fredapi as fp

# Organization
Data loading, CPI adjustments, and dataset preparation.

# Preprocessing: 2010-2012 Data Integration
Convert UCOP format (2010-2012) to match Transparent California format (2013+).

In [ ]:
# Process 2010-2012 UCOP data files
import os
import pandas as pd

notebook_dir = '/Users/joshkenworthy/Library/CloudStorage/OneDrive-UCLAITServices/UC Salary Data'

# Location mapping from city names to full university names
location_map = {
    'Berkeley': 'University of California, Berkeley',
    'Davis': 'University of California, Davis',
    'Irvine': 'University of California, Irvine',
    'Los Angeles': 'University of California, Los Angeles',
    'Merced': 'University of California, Merced',
    'Riverside': 'University of California, Riverside',
    'San Diego': 'University of California, San Diego',
    'San Francisco': 'University of California, San Francisco',
    'Santa Barbara': 'University of California, Santa Barbara',
    'Santa Cruz': 'University of California, Santa Cruz'
}

def preprocess_ucop_file(year):
    """
    Load and preprocess UCOP data file (2010-2012) to match Transparent California format.
    """
    filepath = os.path.join(notebook_dir, f'UCOP_Data_{year}.csv')
    
    # Load data
    df = pd.read_csv(filepath)
    
    # Rename columns to match 2013+ format
    df = df.rename(columns={
        'year': 'Year',
        'location': 'EmployerName',
        'title': 'Position',
        'gross pay': 'TotalWages',
        'regular pay': 'RegularPay',
        'overtime pay': 'OvertimePay',
        'other pay': 'OtherPay'
    })
    
    # Map location names to full university names
    df['EmployerName'] = df['EmployerName'].map(location_map)
    
    # Remove rows with unmapped locations
    df = df[df['EmployerName'].notna()].copy()
    
    # Select only the columns we need (matching the 2013+ required columns)
    required_cols = ['Year', 'EmployerName', 'Position', 'RegularPay', 'TotalWages']
    
    # Add missing columns if needed
    if 'OvertimePay' in df.columns and 'OtherPay' in df.columns:
        # TotalWages should equal RegularPay + OvertimePay + OtherPay
        # Verify and recalculate if necessary
        df['TotalWages'] = df['RegularPay'].fillna(0) + df['OvertimePay'].fillna(0) + df['OtherPay'].fillna(0)
    
    df = df[required_cols].copy()
    
    return df

# Process 2010, 2011, and 2012
for year in [2010, 2011, 2012]:
    df = preprocess_ucop_file(year)
    print(f"Processed {year}: {len(df):,} total records")
    
    # Filter for professors only
    professor_pattern = r'(?:PROFESSOR|Prof[\s\-])'
    df_prof = df[df['Position'].str.contains(professor_pattern, case=False, na=False, regex=True)].copy()
    
    print(f"  → {len(df_prof):,} professor records ({len(df_prof)/len(df)*100:.1f}%)")
    
    # Save professor-only subset
    output_path = os.path.join(notebook_dir, f'uc_professors_{year}.csv')
    df_prof.to_csv(output_path, index=False)
    print(f"  → Saved to {output_path}\n")
    
    # Store in global namespace for later use
    globals()[f'df{str(year)[-2:]}'] = df
    globals()[f'uc_professors_{year}'] = df_prof

print("=" * 60)
print("2010-2012 data preprocessing complete!")
print("=" * 60)

In [ ]:
# Verify 2010-2024 data integration
print("=" * 70)
print("DATA INTEGRATION VERIFICATION")
print("=" * 70)

for year in range(2010, 2025):
    df_name = f"df{str(year)[-2:]}"
    if df_name in globals():
        df = globals()[df_name]
        prof_count = len(df[df['Position'].str.contains(r'Prof[\s\-]', case=False, na=False, regex=True)])
        print(f"{year}: {len(df):,} total | {prof_count:,} professors | Columns: {list(df.columns)[:5]}...")
    else:
        print(f"{year}: NOT LOADED")

print("\n" + "=" * 70)
print(f"Total years loaded: {sum(1 for y in range(2010, 2025) if f'df{str(y)[-2:]}' in globals())}/15")
print("=" * 70)

## Integration Summary

✅ **2010-2012 data successfully integrated!**

The analysis now covers **15 years** (2010-2024) of UC professor salary data.

**Changes made:**
1. Preprocessed UCOP format files (2010-2012) to match Transparent California format
2. Created `uc_professors_2010.csv`, `uc_professors_2011.csv`, `uc_professors_2012.csv`
3. Updated all analysis loops to include years 2010-2024 (was 2013-2024)
4. Location names mapped: "Berkeley" → "University of California, Berkeley", etc.
5. Column names standardized to match 2013+ format

All existing analyses will now include the additional 3 years of historical data.

In [ ]:
# Read professor-only UC salary CSV files
import os
import glob

# Get the directory of the current notebook
notebook_dir = '/Users/joshkenworthy/Library/CloudStorage/OneDrive-UCLAITServices/UC Salary Data'

# Find professor-only CSVs
csv_files = sorted(glob.glob(os.path.join(notebook_dir, 'uc_professors_*.csv')))

# Read each CSV and create dataframes with year-based names
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    year = filename.split('_')[-1].replace('.csv', '')
    year_short = year[-2:]
    
    df_name = f'df{year_short}'
    globals()[df_name] = pd.read_csv(csv_file)
    
    # Compatibility alias for existing analysis logic
    if 'TotalPay' not in globals()[df_name].columns and 'TotalWages' in globals()[df_name].columns:
        globals()[df_name]['TotalPay'] = globals()[df_name]['TotalWages']

In [ ]:
# Configure pay columns (nominal and real)
pay_col = 'TotalPay'
real_pay_col = 'RealTotalPay'
salary_col = real_pay_col
salary_label = 'TotalPay (CPI-adjusted, 2024$)'

# Load CPI inflation rates and build a CPI index
cpi_path = os.path.join(notebook_dir, 'cpi.csv')
cpi = pd.read_csv(cpi_path)
cpi['Year'] = pd.to_datetime(cpi['observation_date']).dt.year
cpi = cpi.sort_values('Year').reset_index(drop=True)
cpi['CPI_Index'] = 100.0
for i in range(1, len(cpi)):
    cpi.loc[i, 'CPI_Index'] = cpi.loc[i - 1, 'CPI_Index'] * (1 + cpi.loc[i, 'CPIAUCSL'] / 100.0)

ref_year = int(cpi['Year'].max())
ref_index = float(cpi.loc[cpi['Year'] == ref_year, 'CPI_Index'].iloc[0])
cpi_index = dict(zip(cpi['Year'], cpi['CPI_Index']))

import re

def normalize_col(col):
    return re.sub(r'[^a-z0-9]+', '', col.lower())

def resolve_totalpay_column(df):
    normalized = {col: normalize_col(col) for col in df.columns}
    for col, norm in normalized.items():
        if norm == 'totalpay':
            return col, 'Exact'
    for col, norm in normalized.items():
        if 'totalpay' in norm and 'benefits' not in norm:
            return col, 'Partial'
    for col, norm in normalized.items():
        if norm == 'totalpaybenefits':
            return col, 'Fallback:TotalPayBenefits'
    return None, 'Not Found'

removed_summary = []
pay_components = ['RegularPay', 'OvertimePay', 'LumpSumPay', 'OtherPay']

for year in range(10, 25):
    df_name = f"df{year:02d}"
    if df_name in globals():
        df = globals()[df_name]
        year_full = 2000 + year
        if pay_col in df.columns:
            resolved_pay_col, resolution = pay_col, 'Exact'
        else:
            resolved_pay_col, resolution = resolve_totalpay_column(df)
            if resolved_pay_col is None and all(col in df.columns for col in pay_components):
                df[pay_col] = df[pay_components].fillna(0).sum(axis=1)
                resolved_pay_col, resolution = pay_col, 'Computed from components'
        if resolved_pay_col and year_full in cpi_index:
            df = df[df[resolved_pay_col] > 0].copy()
            deflator = ref_index / cpi_index[year_full]
            df[real_pay_col] = df[resolved_pay_col] * deflator
            globals()[df_name] = df
            removed_summary.append({
                'Year': str(year_full),
                'Pay Col': resolved_pay_col,
                'Resolution': resolution
            })
        else:
            removed_summary.append({
                'Year': str(year_full),
                'Pay Col': resolved_pay_col if resolved_pay_col else 'Not Found',
                'Resolution': resolution
            })

In [ ]:
# Build professor-only subsets (2010–2024) with required columns only
# Uses the same professor definition already implemented in this notebook.
professor_pattern = r'(?:PROFESSOR|Prof[\s\-])'
required_cols = ['Year', 'EmployerName', 'Position', 'RegularPay', 'TotalWages']

for year in range(2010, 2025):
    df_name = f"df{str(year)[-2:]}"
    if df_name in globals():
        df_year = globals()[df_name]
        prof_subset = df_year[df_year['Position'].str.contains(professor_pattern, case=False, na=False, regex=True)].copy()
        prof_subset['Year'] = year
        uc_df_name = f"uc_professors_{year}"
        globals()[uc_df_name] = prof_subset[required_cols].copy()

In [ ]:
# Validate professor subsets before exporting
required_cols = ['Year', 'EmployerName', 'Position', 'RegularPay', 'TotalWages']
professor_pattern = r'(?:PROFESSOR|Prof[\s\-])'

validation_rows = []
all_ok = True

for year in range(2010, 2025):
    df_name = f"uc_professors_{year}"
    if df_name in globals():
        df = globals()[df_name]
        cols_ok = list(df.columns) == required_cols
        pattern_ok = df['Position'].str.contains(professor_pattern, case=False, na=False, regex=True).all()
        year_ok = (df['Year'] == year).all()
        row_count = len(df)
        validation_rows.append({
            'Year': year,
            'Rows': row_count,
            'Cols OK': cols_ok,
            'Prof Pattern OK': pattern_ok,
            'Year OK': year_ok
        })
        all_ok = all_ok and cols_ok and pattern_ok and year_ok
    else:
        validation_rows.append({
            'Year': year,
            'Rows': None,
            'Cols OK': False,
            'Prof Pattern OK': False,
            'Year OK': False
        })
        all_ok = False

validation_df = pd.DataFrame(validation_rows)

# Analysis: University Breakdown
Descriptive statistics comparing the UC system campuses.

In [ ]:
# Define rank categorization before university analysis
def categorize_rank(position):
    pos_lower = position.lower()
    if 'asst' in pos_lower or 'assistant' in pos_lower:
        return 'Assistant'
    elif 'assoc' in pos_lower:
        return 'Associate'
    elif 'clin' in pos_lower:
        return 'Clinical'
    elif 'res' in pos_lower or 'research' in pos_lower or 'in res' in pos_lower:
        return 'Research'
    return 'Full'

In [ ]:
# Prepare 2024 data for university analysis
prof_2024 = df24[df24['Position'].str.contains(r'Prof[\s\-]', case=False, na=False, regex=True)].copy()
prof_2024['Rank'] = prof_2024['Position'].apply(categorize_rank)
prof_2024['TotalPay'] = prof_2024.get('TotalPay', prof_2024.get('TotalWages', 0))

# 1. Total Professor Count by University
univ_counts = prof_2024.groupby('EmployerName').size().sort_values(ascending=False).reset_index(name='TotalProfessors')

# 2. Rank Distribution by University (both counts and percentages)
rank_by_univ = pd.crosstab(prof_2024['EmployerName'], prof_2024['Rank'])
rank_by_univ_pct = pd.crosstab(prof_2024['EmployerName'], prof_2024['Rank'], normalize='columns') * 100
rank_by_univ_pct = rank_by_univ_pct.round(1)

# 3. Salary Quartiles by University
def get_quartiles(series):
    return pd.Series({
        'Q1 (25%)': series.quantile(0.25),
        'Median (50%)': series.quantile(0.50),
        'Q3 (75%)': series.quantile(0.75),
        'Mean': series.mean(),
        'Std Dev': series.std()
    })

salary_quartiles = prof_2024.groupby('EmployerName')['TotalPay'].apply(get_quartiles).round(0)

# 4. Compensation by Rank within each University
salary_by_rank_univ = prof_2024.pivot_table(values='TotalPay', index='EmployerName', columns='Rank', aggfunc='mean').round(0)

# 5. Identify notable differences (highest/lowest mean by rank across universities)
rank_var = prof_2024.groupby(['Rank', 'EmployerName'])['TotalPay'].mean().unstack()
rank_ranges = pd.DataFrame({
    'Highest': rank_var.max(axis=1),
    'Lowest': rank_var.min(axis=1),
    'Range': rank_var.max(axis=1) - rank_var.min(axis=1),
    'Range %': ((rank_var.max(axis=1) - rank_var.min(axis=1)) / rank_var.min(axis=1) * 100).round(1)
})

## Visualization: University Breakdown

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Professor Count by University
ax1 = axes[0, 0]
colors_univ = ['#005581' if i < 3 else '#3a7ca5' if i < 6 else '#6fb3d8' for i in range(len(univ_counts))]
ax1.barh(range(len(univ_counts)), univ_counts['TotalProfessors'], color=colors_univ)
ax1.set_yticks(range(len(univ_counts)))
ax1.set_yticklabels([name.replace('University of California, ', '').replace('San ', 'S. ') for name in univ_counts['EmployerName']], fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel('Number of Professors')
ax1.set_title('Total Professors by University (2024)', fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# 2. Rank Distribution by University (Stacked Bar)
ax2 = axes[0, 1]
rank_colors = {'Assistant': '#005581', 'Associate': '#3a7ca5', 'Full': '#e67e22', 'Clinical': '#c13d3d', 'Research': '#27ae60'}
rank_by_univ_sorted = rank_by_univ.loc[univ_counts['EmployerName']]
rank_by_univ_sorted.plot(kind='barh', stacked=True, ax=ax2, color=[rank_colors.get(rank, '#999') for rank in rank_by_univ_sorted.columns])
ax2.set_xlabel('Number of Professors')
ax2.set_title('Rank Distribution by University', fontweight='bold')
ax2.set_yticklabels([name.replace('University of California, ', '').replace('San ', 'S. ') for name in rank_by_univ_sorted.index], fontsize=10)
ax2.legend(title='Rank', loc='lower right', fontsize=9)
ax2.grid(axis='x', alpha=0.3)

# 3. Salary Range by University (Box plot)
ax3 = axes[1, 0]
box_data = [prof_2024[prof_2024['EmployerName'] == univ]['TotalPay'].values for univ in univ_counts['EmployerName']]
bp = ax3.boxplot(box_data, labels=[name.replace('University of California, ', '').replace('San ', 'S. ') for name in univ_counts['EmployerName']], vert=True, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#005581')
    patch.set_alpha(0.7)
ax3.set_ylabel('Total Pay ($)')
ax3.set_title('Salary Distribution by University', fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(axis='y', alpha=0.3)

# 4. Compensation Range by Rank (showing variation across universities)
ax4 = axes[1, 1]
rank_order = ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']
x_pos = np.arange(len(rank_order))
ranges = [rank_ranges.loc[rank, 'Range'] if rank in rank_ranges.index else 0 for rank in rank_order]
range_pcts = [rank_ranges.loc[rank, 'Range %'] if rank in rank_ranges.index else 0 for rank in rank_order]

bars = ax4.bar(x_pos, ranges, color=['#005581', '#3a7ca5', '#e67e22', '#c13d3d', '#27ae60'])
ax4.set_xticks(x_pos)
ax4.set_xticklabels(rank_order, rotation=45, ha='right')
ax4.set_ylabel('Salary Range ($)')
ax4.set_title('Compensation Range by Rank Across Universities', fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# Add percentage labels on bars
for i, (bar, pct) in enumerate(zip(bars, range_pcts)):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{pct:.0f}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Salary heatmap by rank across universities
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# 1. Mean salary by rank and university (heatmap)
ax1 = axes[0]
salary_heatmap = prof_2024.pivot_table(values='TotalPay', index='EmployerName', columns='Rank', aggfunc='mean')
salary_heatmap = salary_heatmap.loc[univ_counts['EmployerName']]  # Sort by university size
im = ax1.imshow(salary_heatmap.values, cmap='YlOrRd', aspect='auto')
ax1.set_xticks(range(len(salary_heatmap.columns)))
ax1.set_yticks(range(len(salary_heatmap.index)))
ax1.set_xticklabels(salary_heatmap.columns, fontsize=11)
ax1.set_yticklabels([name.replace('University of California, ', '').replace('San ', 'S. ') for name in salary_heatmap.index], fontsize=10)
ax1.set_title('Mean Salary Heatmap by Rank and University', fontweight='bold', fontsize=12)

# Add colorbar
cbar = plt.colorbar(im, ax=ax1)
cbar.set_label('Mean Salary ($)', fontsize=10)

# Add salary values in cells
for i in range(len(salary_heatmap.index)):
    for j in range(len(salary_heatmap.columns)):
        val = salary_heatmap.values[i, j]
        if not np.isnan(val):
            text_color = 'white' if val > salary_heatmap.values.mean() else 'black'
            ax1.text(j, i, f'${val/1000:.0f}K', ha='center', va='center', color=text_color, fontsize=8, fontweight='bold')

# 2. Coefficient of Variation (salary disparity within rank by university)
ax2 = axes[1]
cv_by_rank_univ = prof_2024.groupby(['Rank', 'EmployerName'])['TotalPay'].apply(lambda x: x.std() / x.mean()).unstack()
cv_by_rank_univ = cv_by_rank_univ.loc[:, univ_counts['EmployerName']]

x_pos = np.arange(len(cv_by_rank_univ.index))
width = 0.15
rank_order_sorted = ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']
colors_cv = ['#005581', '#3a7ca5', '#e67e22', '#c13d3d', '#27ae60']

for i, univ in enumerate(cv_by_rank_univ.columns):
    if i < 5:  # Plot top 5 universities only for clarity
        cv_values = [cv_by_rank_univ.loc[rank, univ] if rank in cv_by_rank_univ.index else 0 for rank in rank_order_sorted]
        ax2.bar(x_pos + i*width, cv_values, width, label=univ.replace('University of California, ', '').replace('San ', 'S. '), alpha=0.8)

ax2.set_xlabel('Rank', fontsize=11)
ax2.set_ylabel('Coefficient of Variation (CV)', fontsize=11)
ax2.set_title('Salary Disparity Within Rank (Coefficient of Variation)', fontweight='bold', fontsize=12)
ax2.set_xticks(x_pos + width * 2)
ax2.set_xticklabels(rank_order_sorted, fontsize=10)
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution comparison: median vs mean salary by university and rank
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 1. Mean salary progression by rank across top universities
ax1 = axes[0]
rank_order_sorted = ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']
top_univs = univ_counts['EmployerName'].head(6)
colors_line = ['#005581', '#3a7ca5', '#e67e22', '#c13d3d', '#27ae60', '#16a085']

for idx, univ in enumerate(top_univs):
    univ_data = prof_2024[prof_2024['EmployerName'] == univ]
    means = [univ_data[univ_data['Rank'] == rank]['TotalPay'].mean() if rank in univ_data['Rank'].values else None for rank in rank_order_sorted]
    ax1.plot(range(len(rank_order_sorted)), means, marker='o', linewidth=2.5, markersize=8, label=univ.replace('University of California, ', '').replace('San ', 'S. '), color=colors_line[idx])

ax1.set_xticks(range(len(rank_order_sorted)))
ax1.set_xticklabels(rank_order_sorted, fontsize=11)
ax1.set_ylabel('Mean Salary ($)', fontsize=11)
ax1.set_title('Salary Progression by Rank - Top 6 Universities', fontweight='bold', fontsize=12)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# 2. Salary percentile bands by university (quartiles)
ax2 = axes[1]
univs_to_plot = univ_counts['EmployerName'].head(8)
univ_labels = [name.replace('University of California, ', '').replace('San ', 'S. ') for name in univs_to_plot]

percentiles_data = []
for univ in univs_to_plot:
    univ_data = prof_2024[prof_2024['EmployerName'] == univ]['TotalPay']
    percentiles_data.append([
        univ_data.quantile(0.1),
        univ_data.quantile(0.25),
        univ_data.quantile(0.50),
        univ_data.quantile(0.75),
        univ_data.quantile(0.90)
    ])

percentiles_data = np.array(percentiles_data)
x_pos = np.arange(len(univs_to_plot))

# Create violin-like visualization using percentile bands
ax2.fill_between(x_pos - 0.3, percentiles_data[:, 0], percentiles_data[:, 4], alpha=0.2, color='#005581', label='10th-90th percentile')
ax2.fill_between(x_pos - 0.3, percentiles_data[:, 1], percentiles_data[:, 3], alpha=0.5, color='#005581', label='25th-75th percentile')
ax2.plot(x_pos - 0.3, percentiles_data[:, 2], marker='D', color='#c13d3d', linewidth=2, markersize=8, label='Median')

ax2.set_xticks(x_pos - 0.3)
ax2.set_xticklabels(univ_labels, rotation=45, ha='right', fontsize=10)
ax2.set_ylabel('Salary ($)', fontsize=11)
ax2.set_title('Salary Distribution by University (Percentile Bands)', fontweight='bold', fontsize=12)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

# Analysis: 2024 Snapshot
Focused analysis on the 2024 professor subset.

In [ ]:
# Create subset with positions that have "Prof " (with space) or "Prof-" to exclude "Professional"
prof_subset_24 = df24[df24['Position'].str.contains(r'Prof[\s\-]', case=False, na=False, regex=True)]

In [ ]:
# Salary Analysis for Professor Positions in 2024
print("=" * 70)
print("PROFESSOR SALARY ANALYSIS - 2024 DATA")
print("=" * 70)

print(f"\nUsing column: '{salary_col}'")
print(f"Total professors: {len(prof_subset_24)}")
print(f"Non-zero {pay_col} records: {(prof_subset_24[pay_col] > 0).sum()}")

print("\n1. SALARY STATISTICS")
print("-" * 70)
print(prof_subset_24[salary_col].describe())

# Group by position type
print("\n2. SALARY BY POSITION TYPE (Top 15)")
print("-" * 70)
position_stats = prof_subset_24.groupby('Position')[salary_col].agg(['count', 'mean', 'median', 'min', 'max'])
position_stats = position_stats.sort_values('mean', ascending=False)
print(position_stats.head(15))

# Check for salary by campus/organization
print(f"\n3. SALARY BY EMPLOYER")
print("-" * 70)
org_stats = prof_subset_24.groupby('EmployerName')[salary_col].agg(['count', 'mean', 'median'])
org_stats = org_stats.sort_values('mean', ascending=False)
print(org_stats.head(10))

# Visualize salary distribution
print("\n4. SALARY DISTRIBUTION VISUALIZATION")
print("-" * 70)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
prof_subset_24[salary_col].hist(bins=50, ax=axes[0, 0], edgecolor='black', color='steelblue')
axes[0, 0].set_title(f'{salary_label} Distribution (Histogram)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel(salary_label)
axes[0, 0].set_ylabel('Frequency')

# Box plot by top 20 positions
top_positions = prof_subset_24['Position'].value_counts().head(20).index
prof_subset_top = prof_subset_24[prof_subset_24['Position'].isin(top_positions)]
prof_subset_top.boxplot(column=salary_col, by='Position', ax=axes[0, 1])
axes[0, 1].set_title(f'{salary_label} by Top 20 Position Types', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Position')
axes[0, 1].set_ylabel(salary_label)
plt.setp(axes[0, 1].xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

# Top 10 highest paid
top_10 = prof_subset_24.nlargest(10, salary_col)[[salary_col, 'Position', 'EmployerName']]
axes[1, 0].barh(range(len(top_10)), top_10[salary_col].values, color='darkgreen')
axes[1, 0].set_yticks(range(len(top_10)))
axes[1, 0].set_yticklabels([f"{pos[:30]}" for pos in top_10['Position'].values], fontsize=9)
axes[1, 0].set_title(f'Top 10 Highest {salary_label}', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel(salary_label)
axes[1, 0].ticklabel_format(style='plain', axis='x')

# Salary percentiles
percentiles = [10, 25, 50, 75, 90, 95, 99]
percentile_values = [np.percentile(prof_subset_24[salary_col].dropna(), p) for p in percentiles]
axes[1, 1].plot(percentiles, percentile_values, marker='o', linewidth=2, markersize=8, color='darkred')
axes[1, 1].set_title(f'{salary_label} Percentiles', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Percentile')
axes[1, 1].set_ylabel(salary_label)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

# Multi-Year Analysis
Professor salary trends across 2010–2024.

In [ ]:
# Collect professor salary data across all years (refined regex: "Prof " or "Prof-" only)
print("=" * 70)
print("PROFESSOR SALARY TRENDS ACROSS ALL YEARS (2010-2024)")
print("=" * 70)

# Create professor subsets for all years using refined pattern
prof_subsets = {}
years = list(range(10, 25))  # 10-24 for 2010-2024

for year in years:
    df_name = f'df{year:02d}'
    if df_name in globals():
        df = globals()[df_name]
        # Use regex to match "Prof " (with space) or "Prof-" to exclude "Professional"
        prof_subset = df[df['Position'].str.contains(r'Prof[\s\-]', case=False, na=False, regex=True)]
        prof_subsets[f'20{year}'] = prof_subset
        print(f"{df_name} -> 20{year}: {len(prof_subset)} professors")

print("\n" + "=" * 70)
print("YEAR-BY-YEAR STATISTICS")
print("=" * 70)

# Collect statistics for each year
yearly_stats = []
for year_str in sorted(prof_subsets.keys()):
    df_prof = prof_subsets[year_str]
    stats = {
        'Year': year_str,
        'Count': len(df_prof),
        'Mean': df_prof[salary_col].mean(),
        'Median': df_prof[salary_col].median(),
        'Std': df_prof[salary_col].std(),
        'Min': df_prof[salary_col].min(),
        'Max': df_prof[salary_col].max(),
        'Q25': df_prof[salary_col].quantile(0.25),
        'Q75': df_prof[salary_col].quantile(0.75),
    }
    yearly_stats.append(stats)

yearly_df = pd.DataFrame(yearly_stats)
print(yearly_df.to_string(index=False))

print("\n" + "=" * 70)
print("YEAR-OVER-YEAR CHANGE IN MEAN SALARY")
print("=" * 70)
yearly_df['YoY_Change'] = yearly_df['Mean'].diff()
yearly_df['YoY_Change_Pct'] = yearly_df['Mean'].pct_change() * 100
print(yearly_df[['Year', 'Mean', 'YoY_Change', 'YoY_Change_Pct']].to_string(index=False))

print("\n" + "=" * 70)
print("ANALYSIS: Changes Starting in 2010 (Public Data Availability)")
print("=" * 70)

mean_2010 = yearly_df[yearly_df['Year'] == '2010']['Mean'].values[0]
mean_2019 = yearly_df[yearly_df['Year'] == '2019']['Mean'].values[0]
mean_2024 = yearly_df[yearly_df['Year'] == '2024']['Mean'].values[0]

change_2010_to_2019 = mean_2019 - mean_2010
change_2010_to_2024 = mean_2024 - mean_2010
pct_change_2010_to_2019 = (change_2010_to_2019 / mean_2010) * 100 if mean_2010 > 0 else 0
pct_change_2010_to_2024 = (change_2010_to_2024 / mean_2010) * 100 if mean_2010 > 0 else 0

print(f"\n2010 Mean Salary (Baseline): ${mean_2010:,.2f}")
print(f"2019 Mean Salary: ${mean_2019:,.2f}")
print(f"2024 Mean Salary: ${mean_2024:,.2f}")
print(f"\n2010 → 2019 Change: ${change_2010_to_2019:,.2f} ({pct_change_2010_to_2019:+.2f}%)")
print(f"2010 → 2024 Total Change: ${change_2010_to_2024:,.2f} ({pct_change_2010_to_2024:+.2f}%)")

# Historical trends post-2010
avg_2011_2024 = yearly_df[yearly_df['Year'] > '2010']['Mean'].mean()
std_2011_2024 = yearly_df[yearly_df['Year'] > '2010']['Mean'].std()

print(f"\nAverage Mean Salary (2011-2024): ${avg_2011_2024:,.2f}")
print(f"Std Dev (2011-2024): ${std_2011_2024:,.2f}")
print(f"2010 vs Post-2010 Avg Difference: ${avg_2011_2024 - mean_2010:,.2f} ({((avg_2011_2024 - mean_2010) / std_2011_2024):.2f} std devs)")

# Visualization: Trends
Year-by-year trend charts for professor compensation.

In [ ]:
# Visualize salary trends across years
print("\n" + "=" * 70)
print("SALARY TREND VISUALIZATION")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Extract years for plotting
years_list = yearly_df['Year'].tolist()
years_numeric = [int(y) for y in years_list]

# 1. Mean salary trend
axes[0, 0].plot(years_numeric, yearly_df['Mean'], marker='o', linewidth=2.5, markersize=8, color='steelblue')
axes[0, 0].axvline(x=2010, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='2010 Baseline')
axes[0, 0].fill_between(years_numeric, yearly_df['Mean'] - yearly_df['Std'], yearly_df['Mean'] + yearly_df['Std'], alpha=0.2, color='steelblue')
axes[0, 0].set_title(f'Mean Professor {salary_label} Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel(salary_label)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()
axes[0, 0].ticklabel_format(style='plain', axis='y')

# 2. Year-over-year percentage change
yoy_change_pct = yearly_df['YoY_Change_Pct'].fillna(0)
colors = ['green' if x > 0 else 'red' for x in yoy_change_pct]
axes[0, 1].bar(years_numeric, yoy_change_pct, color=colors, alpha=0.7, edgecolor='black')
axes[0, 1].axvline(x=2010, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='2010 Baseline')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].set_title('Year-over-Year Salary Change (%)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('YoY Change (%)')
axes[0, 1].grid(True, alpha=0.3, axis='y')
axes[0, 1].legend()

# 3. Median + Quartiles
axes[1, 0].plot(years_numeric, yearly_df['Median'], marker='s', linewidth=2.5, markersize=8, color='darkgreen', label='Median')
axes[1, 0].plot(years_numeric, yearly_df['Q25'], marker='^', linewidth=2, markersize=6, color='orange', label='Q25')
axes[1, 0].plot(years_numeric, yearly_df['Q75'], marker='^', linewidth=2, markersize=6, color='purple', label='Q75')
axes[1, 0].axvline(x=2010, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='2010 Baseline')
axes[1, 0].set_title('Median and Quartile Trends', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel(salary_label)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()
axes[1, 0].ticklabel_format(style='plain', axis='y')

# 4. Count of professors over time
axes[1, 1].plot(years_numeric, yearly_df['Count'], marker='d', linewidth=2.5, markersize=8, color='darkred')
axes[1, 1].axvline(x=2010, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='2010 Baseline')
axes[1, 1].set_title('Number of Professors Over Time', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Count')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n✓ Visualizations complete!")

# Analysis: Rank Variation
Variation and dispersion metrics by professor rank.

In [ ]:
# Categorize professor ranks based on title
def categorize_rank(position):
    pos_lower = position.lower()
    if 'assist' in pos_lower:
        return 'Assistant'
    elif 'assoc' in pos_lower:
        return 'Associate'
    elif 'clinical' in pos_lower or 'clin prof' in pos_lower:
        return 'Clinical'
    elif 'research' in pos_lower or 'res prof' in pos_lower:
        return 'Research'
    elif 'prof' in pos_lower:
        return 'Full'
    return 'Other'

for year_str in prof_subsets.keys():
    prof_subsets[year_str]['Rank'] = prof_subsets[year_str]['Position'].apply(categorize_rank)

print("=" * 80)
print("SALARY VARIATION WITHIN PROFESSOR RANKS BY YEAR")
print("=" * 80)

# Analyze variation metrics by rank and year
variation_analysis = []

for year_str in sorted(prof_subsets.keys()):
    df_year = prof_subsets[year_str]
    year_int = int(year_str)
    
    for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
        rank_data = df_year[df_year['Rank'] == rank][salary_col]
        
        if len(rank_data) > 0:
            mean_sal = rank_data.mean()
            std_sal = rank_data.std()
            cv = (std_sal / mean_sal * 100) if mean_sal > 0 else 0  # Coefficient of Variation
            min_sal = rank_data.min()
            max_sal = rank_data.max()
            range_sal = max_sal - min_sal
            q25 = rank_data.quantile(0.25)
            q75 = rank_data.quantile(0.75)
            iqr = q75 - q25
            count = len(rank_data)
            
            variation_analysis.append({
                'Year': year_int,
                'Rank': rank,
                'Count': count,
                'Mean': mean_sal,
                'Std': std_sal,
                'CV%': cv,
                'Min': min_sal,
                'Max': max_sal,
                'Range': range_sal,
                'Q25': q25,
                'Q75': q75,
                'IQR': iqr,
            })

variation_df = pd.DataFrame(variation_analysis)

# Display variation metrics for each rank
for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
    rank_data = variation_df[variation_df['Rank'] == rank]
    if len(rank_data) > 0:
        print(f"\n{rank.upper()} PROFESSORS")
        print("-" * 80)
        print(rank_data[['Year', 'Count', 'Mean', 'Std', 'CV%', 'IQR']].to_string(index=False))
        
        # Calculate trends: 2010 baseline vs 2011-2024
        years_in_data = sorted(rank_data['Year'].unique())
        if len(years_in_data) > 1:
            baseline_2010 = rank_data[rank_data['Year'] == 2010]['CV%'].mean() if 2010 in years_in_data else None
            cv_2011_2024 = rank_data[rank_data['Year'] > 2010]['CV%'].mean()
            
            std_2010 = rank_data[rank_data['Year'] == 2010]['Std'].values
            std_2024 = rank_data[rank_data['Year'] == 2024]['Std'].values
            
            if len(std_2010) > 0 and len(std_2024) > 0:
                std_change_pct = ((std_2024[0] - std_2010[0]) / std_2010[0] * 100) if std_2010[0] > 0 else 0
                if baseline_2010 is not None:
                    print(f"\n  2010 Baseline CV: {baseline_2010:.2f}%")
                print(f"  Avg CV (2011-2024): {cv_2011_2024:.2f}%")
                print(f"  Std Dev Change (2010→2024): {std_change_pct:+.2f}%")

# Visualization: Rank Variation
Visual trends in variation metrics by rank.

In [ ]:
# Visualize salary variation trends by rank
print("\n" + "=" * 80)
print("VISUALIZATION: SALARY VARIATION TRENDS BY RANK")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Coefficient of Variation (CV) over time by rank
for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
    rank_data = variation_df[variation_df['Rank'] == rank].sort_values('Year')
    if len(rank_data) > 0:
        axes[0, 0].plot(rank_data['Year'], rank_data['CV%'], marker='o', label=rank, linewidth=2)

axes[0, 0].axvline(x=2010, color='blue', linestyle='--', alpha=0.7, linewidth=2, label='2010 Baseline')
axes[0, 0].set_title('Coefficient of Variation by Rank (Lower = Less Dispersion)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('CV (%)')
axes[0, 0].legend(loc='best')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Standard Deviation over time by rank
for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
    rank_data = variation_df[variation_df['Rank'] == rank].sort_values('Year')
    if len(rank_data) > 0:
        axes[0, 1].plot(rank_data['Year'], rank_data['Std'], marker='s', label=rank, linewidth=2)

axes[0, 1].axvline(x=2010, color='blue', linestyle='--', alpha=0.7, linewidth=2, label='2010 Baseline')
axes[0, 1].set_title('Standard Deviation of Salary by Rank', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel(salary_label)
axes[0, 1].legend(loc='best')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].ticklabel_format(style='plain', axis='y')

# Plot 3: Interquartile Range (IQR) over time by rank
for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
    rank_data = variation_df[variation_df['Rank'] == rank].sort_values('Year')
    if len(rank_data) > 0:
        axes[1, 0].plot(rank_data['Year'], rank_data['IQR'], marker='^', label=rank, linewidth=2)

axes[1, 0].axvline(x=2010, color='blue', linestyle='--', alpha=0.7, linewidth=2, label='2010 Baseline')
axes[1, 0].set_title('Interquartile Range (IQR) by Rank', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel(salary_label)
axes[1, 0].legend(loc='best')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].ticklabel_format(style='plain', axis='y')

# Plot 4: CV change comparing 2010 baseline vs 2011-2024
cv_change_data = []
ranks_with_data = []

for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
    baseline_2010 = variation_df[(variation_df['Rank'] == rank) & (variation_df['Year'] == 2010)]['CV%'].mean()
    cv_2011_2024 = variation_df[(variation_df['Rank'] == rank) & (variation_df['Year'] > 2010)]['CV%'].mean()
    
    if not np.isnan(baseline_2010) and not np.isnan(cv_2011_2024):
        change = ((cv_2011_2024 - baseline_2010) / baseline_2010 * 100)
        cv_change_data.append(change)
        ranks_with_data.append(rank)

colors = ['green' if x < 0 else 'red' for x in cv_change_data]
axes[1, 1].barh(ranks_with_data, cv_change_data, color=colors, alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=0, color='black', linestyle='-', linewidth=1)
axes[1, 1].set_title('CV Change: 2010 Baseline vs 2011-2024\n(Negative = Compression)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('% Change in CV')
axes[1, 1].set_ylabel('Rank')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Statistical Test: Levene's Variance Test
Assess variance changes: 2010 baseline vs 2011–2024.

In [ ]:
# Statistical test: Levene's test for equality of variances: 2010 baseline vs 2011-2024
from scipy import stats

print("\n" + "=" * 80)
print("STATISTICAL TEST: LEVENE'S TEST FOR VARIANCE EQUALITY")
print("Testing if salary variation within each rank changed from 2010 baseline")
print("=" * 80)

results_levene = []

for rank in ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']:
    # Get 2010 baseline and 2011-2024 data
    baseline_2010_data = []
    post_2010_data = []
    
    for year_str in sorted(prof_subsets.keys()):
        year_int = int(year_str)
        df_year = prof_subsets[year_str]
        rank_salaries = df_year[df_year['Rank'] == rank][salary_col].values
        
        if len(rank_salaries) > 0:
            if year_int == 2010:
                baseline_2010_data.extend(rank_salaries)
            elif year_int > 2010:
                post_2010_data.extend(rank_salaries)
    
    if len(baseline_2010_data) > 1 and len(post_2010_data) > 1:
        # Levene's test
        statistic, p_value = stats.levene(baseline_2010_data, post_2010_data)
        
        # Also calculate variance ratio
        var_baseline = np.var(baseline_2010_data, ddof=1)
        var_post = np.var(post_2010_data, ddof=1)
        var_ratio = var_post / var_baseline
        
        results_levene.append({
            'Rank': rank,
            '2010 Baseline Var': var_baseline,
            '2011-2024 Var': var_post,
            'Var Ratio': var_ratio,
            'Levene Stat': statistic,
            'p-value': p_value,
            'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        })

levene_df = pd.DataFrame(results_levene)
print("\n" + levene_df.to_string(index=False))

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)
print("\nVariance Ratio > 1: Variance INCREASED (2011-2024 more disperse than 2010)")
print("Variance Ratio < 1: Variance DECREASED (2011-2024 more compressed than 2010)")
print("p-value < 0.05: Change in variance is STATISTICALLY SIGNIFICANT")
print("\n" + "-" * 80)

for _, row in levene_df.iterrows():
    rank = row['Rank']
    var_baseline = row['2010 Baseline Var']
    var_post = row['2011-2024 Var']
    p = row['p-value']
    sig = row['Significant (α=0.05)']
    
    print(f"\n{rank} Professors:")
    print(f"  Variance 2010 Baseline: ${var_baseline:,.0f}")
    print(f"  Variance 2011-2024: ${var_post:,.0f}")
    print(f"  Levene's p-value: {p:.4f} ({'Significant' if sig == 'Yes' else 'Not Significant'})")

# Distribution Comparison
2010 baseline vs 2011–2024 salary distributions by rank.

In [ ]:
# Visualize distribution comparison: 2010 baseline vs 2011-2024
print("\n" + "=" * 80)
print("DISTRIBUTION COMPARISON: 2010 BASELINE vs 2011-2024")
print("=" * 80)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Salary Distribution by Rank: 2010 Baseline vs 2011-2024 ({salary_label})', fontsize=14, fontweight='bold')

ranks_to_plot = ['Assistant', 'Associate', 'Full', 'Clinical', 'Research']
colors_baseline = ['steelblue', 'steelblue', 'steelblue', 'steelblue', 'steelblue']
colors_post = ['coral', 'coral', 'coral', 'coral', 'coral']

for idx, rank in enumerate(ranks_to_plot):
    ax = axes.flat[idx]
    
    # Collect 2010 baseline and 2011-2024 data
    baseline_2010_data = []
    post_2010_data = []
    
    for year_str in sorted(prof_subsets.keys()):
        year_int = int(year_str)
        df_year = prof_subsets[year_str]
        rank_salaries = df_year[df_year['Rank'] == rank][salary_col].values
        
        if len(rank_salaries) > 0:
            if year_int == 2010:
                baseline_2010_data.extend(rank_salaries)
            elif year_int > 2010:
                post_2010_data.extend(rank_salaries)
    
    if len(baseline_2010_data) > 0 and len(post_2010_data) > 0:
        # Plot distributions
        ax.hist(baseline_2010_data, bins=50, alpha=0.6, color='steelblue', label='2010 Baseline', edgecolor='black')
        ax.hist(post_2010_data, bins=50, alpha=0.6, color='coral', label='2011-2024', edgecolor='black')
        
        # Add statistics
        mean_baseline = np.mean(baseline_2010_data)
        mean_post = np.mean(post_2010_data)
        std_baseline = np.std(baseline_2010_data)
        std_post = np.std(post_2010_data)
        
        ax.axvline(mean_baseline, color='steelblue', linestyle='--', linewidth=2, label=f'Mean 2010: ${mean_baseline:,.0f}')
        ax.axvline(mean_post, color='coral', linestyle='--', linewidth=2, label=f'Mean Post: ${mean_post:,.0f}')
        
        ax.set_title(f'{rank} Professors\n(2010 Std: ${std_baseline:,.0f}, Post Std: ${std_post:,.0f})', fontweight='bold')
        ax.set_xlabel(salary_label)
        ax.set_ylabel('Frequency')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.ticklabel_format(style='plain', axis='x')

# Remove the 6th subplot (only 5 ranks)
axes.flat[5].axis('off')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Leftward shift in distribution = salary compression")
print("- Narrower distribution = more uniformity within rank")
print("- Overlapping means = similar central tendency")